In [3]:
import json
from pathlib import Path

base = Path.home() / "POPE/output/coco"
out_dir = Path.home() / "learn-to-steer/data/pope/train"
out_dir.mkdir(parents=True, exist_ok=True)


files = {
    "random": base / "coco_pope_random.json",
    "popular": base / "coco_pope_popular.json",
    "adversarial": base / "coco_pope_adversarial.json",
}

merged = []

for subset, path in files.items():
    with open(path, "r") as f:
        for line in f:
            line = line.strip()
            if not line:
                continue
            ex = json.loads(line)
            merged.append({
                "filename": ex["image"],
                "instruction": ex["text"] + " Answer with just one word.",
                "response": ex["label"],
                "subset": subset,
            })

out_path = out_dir / "annotations.json"
with open(out_path, "w") as f:
    json.dump(merged, f, indent=2)

print(f"Saved {len(merged)} examples to {out_path}")

Saved 9000 examples to /research/hal-afsharim/learn-to-steer/data/pope/train/annotations.json


In [ ]:
import json
import random
from pathlib import Path
import numpy as np
import torch

def set_seed(seed_value=0):
    random.seed(seed_value)
    np.random.seed(seed_value)
    torch.manual_seed(seed_value)
    if torch.cuda.is_available():
        torch.cuda.manual_seed(seed_value)
        torch.cuda.manual_seed_all(seed_value)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False
    
    
seed = 0
set_seed(seed)

# Change these paths if needed
pope_root = Path("/research/hal-afsharim/POPE/output/coco")
out_root = Path("/research/hal-afsharim/POPE/pope_split")
coco_val2014 = Path("/research/hal-afsharim/learn-to-steer/data/coco/val2014")

files = {
    "random": pope_root / "coco_pope_random.json",
    "popular": pope_root / "coco_pope_popular.json",
    "adversarial": pope_root / "coco_pope_adversarial.json",
}

splits = {
    "train": [],
    "test": [],
}

for subset, path in files.items():
    subset_data = []

    with open(path, "r") as f:
        for line in f:
            line = line.strip()
            if not line:
                continue
            ex = json.loads(line)  # POPE files are JSONL
            subset_data.append({
                "filename": ex["image"],
                "instruction": ex["text"] + " Answer with just one word.",
                "response": ex["label"],
                "subset": subset,
            })

    random.shuffle(subset_data)

    n = len(subset_data)
    n_train = int(0.8 * n)
    n_test = n - n_train

    splits["train"].extend(subset_data[:n_train])
    splits["test"].extend(subset_data[n_train:n_train + n_test])

# Optional: shuffle within each final split so subsets are mixed
for split_name in splits:
    random.shuffle(splits[split_name])

for split_name, split_data in splits.items():
    split_dir = out_root / split_name
    split_dir.mkdir(parents=True, exist_ok=True)

    with open(split_dir / "annotations.json", "w") as f:
        json.dump(split_data, f, indent=2)

    images_link = split_dir / "images"
    if images_link.is_symlink() or images_link.exists():
        images_link.unlink()
    images_link.symlink_to(coco_val2014)

    print(f"{split_name}: {len(split_data)} samples -> {split_dir / 'annotations.json'}")

print("Per-split subset counts:")
for split_name, split_data in splits.items():
    counts = {}
    for x in split_data:
        counts[x["subset"]] = counts.get(x["subset"], 0) + 1
    print(split_name, counts)

print(f"Total: {sum(len(v) for v in splits.values())}")

train: 7200 samples -> /research/hal-afsharim/POPE/pope_split/train/annotations.json
test: 1800 samples -> /research/hal-afsharim/POPE/pope_split/test/annotations.json
Per-split subset counts:
train {'adversarial': 2400, 'random': 2400, 'popular': 2400}
test {'popular': 600, 'random': 600, 'adversarial': 600}
Total: 9000
